# Notebook 02: Mask R-CNN

**Module:** 09 Instance Segmentation  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Understand Mask R-CNN architecture
2. Know mask head vs box head
3. Run torchvision Mask R-CNN
4. Evaluate with mask IoU

## Mask R-CNN (He et al., 2017)

**Extends Faster R-CNN** with a parallel **mask head**:

```
Image → Backbone → FPN
         ├─ RPN → RoI boxes
         ├─ Box head → class + box refine
         └─ Mask head → K×m×m binary masks (K classes)
```

**Key design:** Predict **class-agnostic** mask (one mask per RoI), then classify separately.

**Mask loss:** Per-pixel sigmoid CE on positive RoIs only (typically 28×28 resolution).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def mask_iou(pred, target):
    """Binary masks (H,W) or (N,H,W). Returns scalar or per-instance IoU."""
    pred = pred.bool()
    target = target.bool()
    if pred.dim() == 2:
        pred, target = pred.unsqueeze(0), target.unsqueeze(0)
    inter = (pred & target).float().sum(dim=(1, 2))
    union = (pred | target).float().sum(dim=(1, 2))
    return inter / (union + 1e-7)

In [ ]:
try:
    from torchvision.models.detection import maskrcnn_resnet50_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

    def build_mask_rcnn(num_classes=2):
        model = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
        in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
        model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
        return model

    model = build_mask_rcnn(num_classes=2)
    model.eval()
    images = [torch.rand(3, 256, 256)]
    with torch.no_grad():
        out = model(images)
    print('Output keys:', out[0].keys())
    print('Boxes:', out[0]['boxes'].shape)
    print('Masks:', out[0]['masks'].shape)  # (N, 1, H, W)
except ImportError as e:
    print('torchvision detection required:', e)

## Mask IoU

Same as box IoU but on pixel masks:

$$\text{maskIoU} = \frac{|M_{pred} \cap M_{gt}|}{|M_{pred} \cup M_{gt}|}$$

COCO instance seg benchmark uses mask AP (AP computed on masks, not boxes).

In [ ]:
# Mask IoU example
pred_m = torch.zeros(64, 64); pred_m[10:30, 10:30] = 1
gt_m = torch.zeros(64, 64); gt_m[12:32, 12:32] = 1
print(f'Mask IoU: {mask_iou(pred_m, gt_m).item():.3f}')

## GeoSpatial: Individual Pond Instance Segmentation

Annotate each pond polygon with unique instance ID → train Mask R-CNN → get separate masks for adjacent ponds.

---

## Summary

Mask R-CNN = Faster R-CNN + FCN mask head per RoI.

**Next:** [03_YOLACT.ipynb](03_YOLACT.ipynb)